# FreshMart Lab 3: Find the best classification model with Automated ML
**Azure Machine Learning (ทางเลือกฉุกเฉินแทน Fabric)**

เส้นหลักของแทร็กนี้: หาโมเดลจำแนก `Churn` ด้วย Automated ML  
ตัวช่วย studio เป็นทางที่แนะนำในห้องเรียน — โน้ตบุ๊กนี้เป็นทาง SDK และทางสำรองเมื่อตัวช่วยใช้ไม่ได้

อ้างอิง Learn: [Find the best classification model with Automated Machine Learning](https://learn.microsoft.com/training/modules/find-best-classification-model-automated-machine-learning/)

### ก่อนรัน
1. ทำแบบฝึกหัด 2 จบแล้ว มี `data/silver/customer_features.csv`
2. รันบน compute instance ของ Azure ML
3. **อย่า Deploy** เป็น endpoint

### ถ้าติด — อ่านก่อนถาม TA
| อาการ | ทำอะไร |
| --- | --- |
| ส่งจ็อบ SDK ไม่ได้ | ใช้เซลล์ทางสำรอง FLAML |
| หาโมเดลไม่เจอ | ตรวจว่าลงทะเบียนชื่อ `freshmart-churn-model` |


### เตรียม loader + ฟังก์ชันฟีเจอร์


In [ ]:
import json
from pathlib import Path
import pandas as pd

from pathlib import Path
import pandas as pd

BRONZE_TRANSACTIONS = "bronze/transactions"
BRONZE_CUSTOMERS = "bronze/customers"
SILVER_CUSTOMER_FEATURES = "silver/customer_features"
GOLD_PREDICTIONS = "gold/freshmart_predictions"
ASSET_BRONZE_TRANSACTIONS = "bronze-transactions"
ASSET_BRONZE_CUSTOMERS = "bronze-customers"
ASSET_SCORING_BATCH = "scoring-batch"
ASSET_SILVER_FEATURES = "silver-customer-features"
ASSET_GOLD_PREDICTIONS = "gold-freshmart-predictions"
TABLE_TO_ASSET = {
    BRONZE_TRANSACTIONS: ASSET_BRONZE_TRANSACTIONS,
    BRONZE_CUSTOMERS: ASSET_BRONZE_CUSTOMERS,
    SILVER_CUSTOMER_FEATURES: ASSET_SILVER_FEATURES,
    GOLD_PREDICTIONS: ASSET_GOLD_PREDICTIONS,
}
CSV_TO_ASSET = {
    "freshmart_transactions.csv": ASSET_BRONZE_TRANSACTIONS,
    "freshmart_customers.csv": ASSET_BRONZE_CUSTOMERS,
    "freshmart_scoring_batch.csv": ASSET_SCORING_BATCH,
}
RAW_FILES = (
    "freshmart_transactions.csv",
    "freshmart_customers.csv",
    "freshmart_scoring_batch.csv",
)


def _first_existing(paths):
    for path in paths:
        candidate = Path(path)
        if candidate.exists() and candidate.is_file():
            return candidate
    return None


def _writable_dir(path: Path) -> Path | None:
    try:
        path.mkdir(parents=True, exist_ok=True)
        probe = path / ".write_test"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink()
        return path.resolve()
    except OSError:
        return None


def resolve_raw_csv(file_name: str) -> Path:
    found = _first_existing([
        Path("data") / "raw" / file_name,
        Path("data") / file_name,
        Path("../data") / "raw" / file_name,
        Path("../data") / file_name,
        Path("../../labs/data") / file_name,
        Path("../labs/data") / file_name,
        Path("labs/data") / file_name,
        Path(file_name),
    ])
    if found is None:
        raise FileNotFoundError(
            f"Cannot find {file_name}. Upload it to data/raw/ next to the notebook "
            "or clone this repo so labs/data/ is available."
        )
    return found


def load_csv(file_name: str) -> pd.DataFrame:
    asset_name = CSV_TO_ASSET.get(file_name)
    if asset_name:
        frame = load_data_asset(asset_name)
        if frame is not None:
            return frame
    found = resolve_raw_csv(file_name)
    print(f"Loaded CSV: {found}")
    return pd.read_csv(found)


def load_data_asset(asset_name: str):
    """Load a registered Azure ML data asset, or return None."""
    try:
        from azure.ai.ml import MLClient
        from azure.identity import DefaultAzureCredential

        ml_client = MLClient.from_config(credential=DefaultAzureCredential())
        asset = ml_client.data.get(name=asset_name, label="latest")
        path = asset.path
        print(f"Loaded data asset {asset_name} v{asset.version}: {path}")
        if str(path).lower().endswith(".parquet"):
            return pd.read_parquet(path)
        return pd.read_csv(path)
    except Exception as exc:
        print(f"Data asset '{asset_name}' unavailable ({exc})")
        return None


def register_data_asset(name: str, path, description: str = "") -> bool:
    """Register a file as an Azure ML data asset. Returns True when saved."""
    try:
        from azure.ai.ml import MLClient
        from azure.ai.ml.entities import Data
        from azure.ai.ml.constants import AssetTypes
        from azure.identity import DefaultAzureCredential

        ml_client = MLClient.from_config(credential=DefaultAzureCredential())
        asset = Data(
            name=name,
            path=str(path),
            type=AssetTypes.URI_FILE,
            description=description or f"FreshMart {name}",
        )
        created = ml_client.data.create_or_update(asset)
        print(f"Registered data asset {created.name} v{created.version}")
        return True
    except Exception as exc:
        print(f"Could not register data asset '{name}' ({exc})")
        return False


def resolve_artifact_root() -> Path:
    for candidate in (
        Path("data"),
        Path("../data"),
        Path("labs-azureml/data"),
    ):
        ready = _writable_dir(candidate)
        if ready is not None:
            return ready
    fallback = Path("data")
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback.resolve()


ARTIFACT_ROOT = resolve_artifact_root()


def layer_path(layer: str, stem: str, suffix: str = ".parquet") -> Path:
    folder = ARTIFACT_ROOT / layer
    folder.mkdir(parents=True, exist_ok=True)
    return folder / f"{stem}{suffix}"


def load_table_or_csv(table_name: str, file_name: str) -> pd.DataFrame:
    asset_name = TABLE_TO_ASSET.get(table_name)
    if asset_name:
        frame = load_data_asset(asset_name)
        if frame is not None:
            return frame
    layer, _, stem = table_name.partition("/")
    parquet = layer_path(layer, stem)
    if parquet.exists():
        frame = pd.read_parquet(parquet)
        print(f"Loaded parquet {parquet}: {len(frame):,} rows")
        return frame
    csv_fallback = ARTIFACT_ROOT / layer / f"{stem}.csv"
    if csv_fallback.exists():
        frame = pd.read_csv(csv_fallback)
        print(f"Loaded CSV artifact {csv_fallback}: {len(frame):,} rows")
        return frame
    print(f"Layer file '{table_name}' not found. Falling back to published CSV.")
    return load_csv(file_name)


def save_layer(frame: pd.DataFrame, table_name: str) -> Path:
    layer, _, stem = table_name.partition("/")
    parquet = layer_path(layer, stem)
    try:
        frame.to_parquet(parquet, index=False)
        print(f"Wrote {parquet} ({len(frame):,} rows)")
        return parquet
    except Exception as exc:
        csv_path = layer_path(layer, stem, suffix=".csv")
        frame.to_csv(csv_path, index=False)
        print(f"Parquet unavailable ({exc}). Wrote {csv_path}")
        return csv_path


In [ ]:
from dataclasses import asdict, dataclass
from typing import Any
import logging

logger = logging.getLogger(__name__)

ID_COLUMN = 'CustomerID'
TARGET_COLUMN = 'Churn'
CATEGORICAL_COLUMNS = ('MembershipTier', 'Gender')
IMPUTE_MEDIAN_COLUMNS = ('Age',)
SCALE_COLUMNS = ('MonetaryTotal', 'AvgBasketSize', 'RecencyDays', 'TenureMonths')
DUMMY_COLUMNS = ('MembershipTier_Bronze', 'MembershipTier_Gold', 'MembershipTier_Platinum', 'MembershipTier_Silver', 'Gender_F', 'Gender_M', 'Gender_Other')
PASSTHROUGH_COLUMNS = ('Frequency', 'ComplaintCount')
FEATURE_COLUMNS = ('Age', 'TenureMonths', 'RecencyDays', 'Frequency', 'MonetaryTotal', 'AvgBasketSize', 'ComplaintCount', 'MembershipTier_Bronze', 'MembershipTier_Gold', 'MembershipTier_Platinum', 'MembershipTier_Silver', 'Gender_F', 'Gender_M', 'Gender_Other')
REQUIRED_RAW_COLUMNS = ('CustomerID', 'Age', 'Gender', 'MembershipTier', 'TenureMonths', 'RecencyDays', 'Frequency', 'MonetaryTotal', 'AvgBasketSize', 'ComplaintCount')

@dataclass(frozen=True)
class FeatureParams:
    """Fitted preprocessing parameters reused at scoring time.

    Attributes:
        age_median: Median Age computed from the training customers.
        scale_mins: Per-column minimum used for min-max scaling.
        scale_maxs: Per-column maximum used for min-max scaling.
        dummy_columns: One-hot columns the model expects, in order.
        feature_columns: Final model input columns, in order.
    """

    age_median: float
    scale_mins: dict[str, float]
    scale_maxs: dict[str, float]
    dummy_columns: list[str]
    feature_columns: list[str]

    def to_dict(self) -> dict[str, Any]:
        """Serialize parameters to a JSON-friendly dictionary."""
        return asdict(self)

    @classmethod
    def from_dict(cls, payload: dict[str, Any]) -> FeatureParams:
        """Create parameters from a dictionary.

        Args:
            payload: Mapping produced by ``to_dict``.

        Returns:
            Validated ``FeatureParams``.

        Raises:
            ValueError: If required keys are missing.
        """
        required = {
            "age_median",
            "scale_mins",
            "scale_maxs",
            "dummy_columns",
            "feature_columns",
        }
        missing = required - set(payload)
        if missing:
            raise ValueError(f"FeatureParams missing keys: {sorted(missing)}")
        return cls(
            age_median=float(payload["age_median"]),
            scale_mins={k: float(v) for k, v in payload["scale_mins"].items()},
            scale_maxs={k: float(v) for k, v in payload["scale_maxs"].items()},
            dummy_columns=list(payload["dummy_columns"]),
            feature_columns=list(payload["feature_columns"]),
        )


def validate_raw_customers(df: pd.DataFrame, *, require_target: bool = True) -> pd.DataFrame:
    """Validate and normalize a raw FreshMart customer frame.

    Args:
        df: Raw customer records from CSV or ``bronze.customers``.
        require_target: When True, require the ``Churn`` column.

    Returns:
        Copy with numeric columns coerced.

    Raises:
        ValueError: If required columns are missing.
    """
    required = REQUIRED_RAW_COLUMNS + ((TARGET_COLUMN,) if require_target else ())
    _require_columns(df, required, frame_name="customer frame")
    numeric_cols = (
        "Age",
        "TenureMonths",
        "RecencyDays",
        "Frequency",
        "MonetaryTotal",
        "AvgBasketSize",
        "ComplaintCount",
    )
    if require_target:
        numeric_cols = numeric_cols + (TARGET_COLUMN,)
    return _numeric_copy(df, numeric_cols)


def _require_columns(df: pd.DataFrame, columns: tuple[str, ...], *, frame_name: str) -> None:
    """Raise if expected columns are missing."""
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise ValueError(f"{frame_name} missing required columns: {missing}")


def _numeric_copy(df: pd.DataFrame, columns: tuple[str, ...]) -> pd.DataFrame:
    """Return a copy with selected columns coerced to numeric."""
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def fit_preprocessor(df_raw: pd.DataFrame) -> FeatureParams:
    """Fit imputation and scaling parameters on training customers.

    Args:
        df_raw: Raw customer DataFrame including ``Churn``.

    Returns:
        Fitted parameters that must be reused for scoring.
    """
    df = validate_raw_customers(df_raw, require_target=True)
    age_median = float(df["Age"].median())
    if pd.isna(age_median):
        raise ValueError("Cannot fit preprocessor: Age median is NaN")

    scale_mins: dict[str, float] = {}
    scale_maxs: dict[str, float] = {}
    for col in SCALE_COLUMNS:
        col_min = float(df[col].min())
        col_max = float(df[col].max())
        if col_max <= col_min:
            raise ValueError(f"Cannot scale {col}: min={col_min}, max={col_max}")
        scale_mins[col] = col_min
        scale_maxs[col] = col_max

    params = FeatureParams(
        age_median=age_median,
        scale_mins=scale_mins,
        scale_maxs=scale_maxs,
        dummy_columns=list(DUMMY_COLUMNS),
        feature_columns=list(FEATURE_COLUMNS),
    )
    logger.info(
        "Fitted feature params: age_median=%.1f, scale_columns=%s",
        params.age_median,
        list(SCALE_COLUMNS),
    )
    return params


def _one_hot_categories(df: pd.DataFrame) -> pd.DataFrame:
    """One-hot encode membership and gender, keeping the expected columns."""
    encoded = pd.get_dummies(df, columns=list(CATEGORICAL_COLUMNS), drop_first=False)
    for col in DUMMY_COLUMNS:
        if col not in encoded.columns:
            encoded[col] = 0
    return encoded


def _min_max_scale(series: pd.Series, col_min: float, col_max: float) -> pd.Series:
    """Scale a series to [0, 1] using fitted min/max."""
    return (series - col_min) / (col_max - col_min + 1e-6)


def transform_customers(
    df_raw: pd.DataFrame,
    params: FeatureParams,
    *,
    require_target: bool = False,
) -> pd.DataFrame:
    """Apply the fitted FreshMart preprocessing contract.

    Args:
        df_raw: Raw customer records (training or scoring batch).
        params: Parameters from ``fit_preprocessor``.
        require_target: When True, keep and validate ``Churn``.

    Returns:
        Feature frame with ``CustomerID``, model columns, and optional ``Churn``.
    """
    df = validate_raw_customers(df_raw, require_target=require_target)
    work = df.copy()
    work["Age"] = work["Age"].fillna(params.age_median)

    work = _one_hot_categories(work)
    for col in SCALE_COLUMNS:
        work[col] = _min_max_scale(
            work[col],
            params.scale_mins[col],
            params.scale_maxs[col],
        )

    bool_cols = work.select_dtypes(include="bool").columns
    work[bool_cols] = work[bool_cols].astype(int)

    ordered = [ID_COLUMN, *params.feature_columns]
    if require_target or TARGET_COLUMN in work.columns:
        ordered.append(TARGET_COLUMN)
    missing = [col for col in ordered if col not in work.columns]
    if missing:
        raise ValueError(f"Transformed frame missing columns: {missing}")

    result = work[ordered].copy()
    feature_frame = result[list(params.feature_columns)]
    if feature_frame.isna().any().any():
        bad = feature_frame.columns[feature_frame.isna().any()].tolist()
        raise ValueError(f"NaN remaining in feature columns: {bad}")
    return result


def model_matrix(df_features: pd.DataFrame, params: FeatureParams) -> pd.DataFrame:
    """Return the model input matrix in signature order.

    Args:
        df_features: Output of ``transform_customers``.
        params: Fitted parameters.

    Returns:
        DataFrame with only model feature columns.
    """
    _require_columns(df_features, tuple(params.feature_columns), frame_name="feature frame")
    return df_features[list(params.feature_columns)].astype(float)


def save_feature_params(params: FeatureParams, path: str | Path) -> Path:
    """Write feature parameters to a JSON file.

    Args:
        params: Fitted parameters.
        path: Destination JSON path.

    Returns:
        Resolved output path.
    """
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(params.to_dict(), indent=2), encoding="utf-8")
    logger.info("Saved feature params to %s", out)
    return out


def load_feature_params(path: str | Path) -> FeatureParams:
    """Load feature parameters from a JSON file.

    Args:
        path: JSON path written by ``save_feature_params``.

    Returns:
        Fitted parameters.

    Raises:
        FileNotFoundError: If the file does not exist.
    """
    src = Path(path)
    if not src.exists():
        raise FileNotFoundError(f"Feature params not found: {src}")
    payload = json.loads(src.read_text(encoding="utf-8"))
    return FeatureParams.from_dict(payload)


### ขั้นตอนที่ 1: โหลด Silver สำหรับ AutoML

**สิ่งที่ควรเห็น:** 1,500 แถว และมีคอลัมน์ `Churn`  
อย่าใส่ `CustomerID` ลงในเมทริกซ์ฝึก


In [ ]:
df_features = load_table_or_csv(SILVER_CUSTOMER_FEATURES, "freshmart_customers.csv")
if "MembershipTier_Bronze" not in df_features.columns:
    raw = load_table_or_csv(BRONZE_CUSTOMERS, "freshmart_customers.csv")
    params = fit_preprocessor(raw)
    df_features = transform_customers(raw, params, require_target=True)
else:
    params_path = _first_existing([
        ARTIFACT_ROOT / "params" / "feature_params.json",
        Path("data/params/feature_params.json"),
        Path("../data/params/feature_params.json"),
    ])
    params = load_feature_params(params_path) if params_path else fit_preprocessor(
        load_table_or_csv(BRONZE_CUSTOMERS, "freshmart_customers.csv")
    )

X = model_matrix(df_features, params)
y = df_features["Churn"].astype(int)
print(f"Features ({len(X.columns)}):", list(X.columns))
print(f"Rows: {len(X):,}")
assert "CustomerID" not in X.columns
assert len(X) == 1500


### ขั้นตอนที่ 2 (ทางเลือก): ส่งงาน AutoML ด้วย SDK v2

ใช้เมื่อต้องการจ็อบบน Azure ML แบบเดียวกับโมดูล Learn  
ถ้าเซลล์นี้ส่งงานไม่ได้ ให้ข้ามไปทางสำรอง FLAML

ค่าจำกัดงบ: `max_trials=5`, `timeout_minutes=20`, เมตริก `AUC_weighted`


In [ ]:
automl_job_submitted = False
try:
    from azure.ai.ml import MLClient, automl, Input
    from azure.ai.ml.constants import AssetTypes
    from azure.identity import DefaultAzureCredential

    ml_client = MLClient.from_config(credential=DefaultAzureCredential())
    silver_csv = layer_path("silver", "customer_features", suffix=".csv")
    if not silver_csv.exists():
        df_features.to_csv(silver_csv, index=False)

    training_data = Input(type=AssetTypes.URI_FILE, path=str(silver_csv))
    compute_name = None
    for compute in ml_client.compute.list():
        if getattr(compute, "type", "") in {"computeinstance", "ComputeInstance", "amlcompute"}:
            compute_name = compute.name
            break

    classification_job = automl.classification(
        compute=compute_name,
        experiment_name="freshmart-churn-prediction",
        training_data=training_data,
        target_column_name="Churn",
        primary_metric="AUC_weighted",
        n_cross_validations=3,
        enable_model_explainability=False,
        tags={"lab": "freshmart-automl"},
    )
    classification_job.set_limits(
        timeout_minutes=20,
        trial_timeout_minutes=5,
        max_trials=5,
        enable_early_termination=True,
    )
    returned_job = ml_client.jobs.create_or_update(classification_job)
    automl_job_submitted = True
    print("Submitted AutoML job:", returned_job.name)
    print("Monitor at:", getattr(returned_job, "studio_url", "Jobs in Azure ML studio"))
except Exception as exc:
    print(f"SDK AutoML job not submitted ({exc}). Use the studio wizard or the FLAML fallback next.")


### ขั้นตอนที่ 3: ทางสำรอง FLAML (เมื่อตัวช่วยหรือ SDK ใช้ไม่ได้)

งบเวลา 60 วินาที — ได้ Champion ท้องถิ่นเพื่อไปแบบฝึกหัด 4 ได้เมื่อ registry ยังว่าง


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

champion_auc = None
try:
    from flaml import AutoML
    import mlflow
    import mlflow.sklearn
    from mlflow.models.signature import infer_signature

    mlflow.set_experiment("freshmart-churn-prediction")
    automl = AutoML()
    settings = {
        "time_budget": 60,
        "metric": "roc_auc",
        "task": "classification",
        "seed": 42,
        "force_cancel": True,
    }
    with mlflow.start_run(run_name="flaml-automl-fallback") as run:
        automl.fit(X_train, y_train, **settings)
        champion_auc = float(1 - automl.best_loss)
        signature = infer_signature(X_train, automl.predict(X_train))
        mlflow.sklearn.log_model(automl, "model", signature=signature)
        mlflow.log_metric("test_roc_auc", roc_auc_score(y_test, automl.predict_proba(X_test)[:, 1]))
        print("Best config:", automl.best_config)
        print("Best validation AUC:", champion_auc)
        mv = mlflow.register_model(f"runs:/{run.info.run_id}/model", "freshmart-churn-model")
        print(f"Registered {mv.name} version {mv.version}")
except Exception as exc:
    print(f"FLAML/MLflow path unavailable ({exc}). Training a local Random Forest so verification can pass.")
    from sklearn.ensemble import RandomForestClassifier
    model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
    model.fit(X_train, y_train)
    champion_auc = float(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))
    print(f"Local fallback AUC: {champion_auc:.4f}")

if champion_auc is None or champion_auc < 0.70:
    raise AssertionError("Champion AUC ควรสูงกว่า 0.70 สำหรับชุด FreshMart")
print("Lab 3 verification passed")
